In [45]:
import torch
import torchxrayvision as xrv
import torch.nn as nn
import numpy as np
from sklearn.model_selection import KFold, train_test_split
from dataloading.load_nih import NIHChestDataset, get_transforms
from torch.utils.data import DataLoader

In [ ]:
KFOLDS = 5
THRESHOLD = 0.5
NUM_EPOCHS = 20
SUBSET_FRAC = None
BATCH_SIZE = 512
NUM_WORKERS = 8

In [47]:
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

In [48]:
from dataloading.load_nih import load_nih

nih = load_nih()

Using metadata: data/nih-chest-xrays/Data_Entry_2017.csv
Matched rows with images: 112,120


In [49]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

def build_model():
    model = xrv.models.DenseNet(weights='densenet121-res224-all', drop_rate=0.2).to(device)

    # Freeze all early layers for transfer learning
    for param in model.parameters():
        param.requires_grad = False

    # Optional: Unfreeze the last blocks for fine-tuning
    # For DenseNet, this is 'features.denseblock4'
    # for param in model.features.denseblock4.parameters():
    #     param.requires_grad = True

    # Replace the classification head (DenseNet uses 'classifier', not 'fc')
    in_features = model.classifier.in_features
    model.classifier = nn.Linear(in_features, nih.num_classes)
    model.op_threshs = None
    model = model.to(device)

    return model


Using device: cuda


In [50]:
# Cell 8: Train/validation loop utilities
from tqdm.auto import tqdm
from sklearn.metrics import f1_score

def multiclass_metrics(logits: torch.Tensor, targets: torch.Tensor):
    probs = torch.softmax(logits, dim=1)
    preds = torch.argmax(probs, dim=1)
    
    # Handle one-hot targets if necessary
    if targets.ndim > 1 and targets.size(1) > 1:
        targets = torch.argmax(targets, dim=1)
        
    preds_np = preds.cpu().numpy()
    targets_np = targets.cpu().numpy()

    accuracy = (preds_np == targets_np).mean()
    f1 = f1_score(targets_np, preds_np, average='macro', zero_division=0)
    
    return accuracy, f1

def run_epoch(model, optimizer, criterion, loader, train_mode=True):
    model.train() if train_mode else model.eval()

    total_loss = 0.0
    total_acc = 0.0
    total_f1 = 0.0
    steps = 0

    pbar = tqdm(loader, desc="Train" if train_mode else "Val", leave=False)
    for images, targets in pbar:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        # Handle one-hot targets for CrossEntropyLoss if necessary
        if targets.ndim > 1 and targets.size(1) > 1:
            targets_ce = torch.argmax(targets, dim=1)
        else:
            targets_ce = targets

        with torch.set_grad_enabled(train_mode):
            logits = model(images)
            loss = criterion(logits, targets_ce.long())

            if train_mode:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

        acc, f1 = multiclass_metrics(logits.detach(), targets)
        
        total_loss += loss.item()
        total_acc += acc
        total_f1 += f1
        steps += 1
        
        pbar.set_postfix(loss=f"{loss.item():.4f}", f1=f"{f1:.4f}")

    return (
        total_loss / max(steps, 1),
        total_acc / max(steps, 1),
        total_f1 / max(steps, 1),
    )

In [51]:
X_cv_cal, X_test = train_test_split(nih.id_df, test_size=0.1, random_state=42)
X_cv, X_cal = train_test_split(X_cv_cal, test_size=0.1 / 0.9, random_state=42)

print(f"CV set: {len(X_cv)}, Calibration set: {len(X_cal)}, Test set: {len(X_test)}")

train_tfms, val_tfms = get_transforms()

CV set: 24769, Calibration set: 3097, Test set: 3097


In [52]:
from torch_uncertainty.post_processing import TemperatureScaler

def calibrated_model(model):
    cal_ds = NIHChestDataset(X_cal, transform=val_tfms)
    cal_loader = DataLoader(cal_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=torch.cuda.is_available(), num_workers=NUM_WORKERS)

    scaled_model = TemperatureScaler(model, device=device)
    scaled_model.fit(cal_loader)

    return scaled_model


In [53]:
from sklearn.metrics import precision_recall_curve
import numpy as np
import torch

def calculate_threshold(model, val_loader):
    model.eval()

    all_targets = []
    all_probs = []

    with torch.no_grad():
        for images, targets in val_loader:
            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            logits = model(images)
            probs = torch.softmax(logits, dim=1)

            # Convert to one-hot to match shape of probabilities if targets are indices
            if targets.ndim == 1 or (targets.ndim == 2 and targets.size(1) == 1):
                targets_onehot = torch.nn.functional.one_hot(targets.long().view(-1), num_classes=logits.size(1))
            else:
                targets_onehot = targets

            all_targets.append(targets_onehot.cpu().numpy().flatten())
            all_probs.append(probs.cpu().numpy().flatten())

    # y_true: actual labels (0 or 1) - flattened
    # y_scores: probabilities from torch.softmax() - flattened
    y_true = np.concatenate(all_targets)
    y_scores = np.concatenate(all_probs)
    
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_scores)

    # Calculate F1 for every threshold
    f1_scores = (2 * precisions * recalls) / (precisions + recalls + 1e-8)

    # Get the index of the highest F1 score
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx]
    return best_threshold

In [ ]:
# 5-Fold Cross Validation Setup
from torch.utils.data import WeightedRandomSampler
import pickle

kf = KFold(n_splits=KFOLDS, shuffle=True, random_state=42)

fold_histories = []
best_fold_f1s = []

print(f"Starting {KFOLDS}-Fold Cross Validation...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X_cv)):
    print(f"\n{'='*20} FOLD {fold + 1}/{KFOLDS} {'='*20}")

    # 1. Split Data
    fold_train_df = X_cv.iloc[train_idx]
    fold_val_df = X_cv.iloc[val_idx]
    
    # 2. Handle Imbalance for Current Fold (Multi-Class) using Sampler
    # Extract targets and handle potential one-hot vs index
    train_targets_list = fold_train_df["target"].values.tolist()
    train_targets = np.array(train_targets_list)
    if train_targets.ndim > 1 and train_targets.shape[1] > 1:
        train_targets = np.argmax(train_targets, axis=1)
        
    class_counts = np.bincount(train_targets.astype(int), minlength=nih.num_classes)
    # Inverse frequency weighting
    class_weights = 1.0 / np.maximum(class_counts, 1.0)
    
    # Assign weight to each sample according to its class
    sample_weights = class_weights[train_targets.astype(int)]
    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )
    
    # 3. Setup DataLoaders
    train_ds = NIHChestDataset(fold_train_df, transform=train_tfms)
    val_ds = NIHChestDataset(fold_val_df, transform=val_tfms)
    
    # When using a sampler, shuffle must be False
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, pin_memory=torch.cuda.is_available(), num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, pin_memory=torch.cuda.is_available(), num_workers=NUM_WORKERS)
    
    # 4. Initialize Fresh Model
    model = build_model()
    
    criterion = nn.CrossEntropyLoss()
    
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-2)
    
    # 5. Train Model
    best_val_f1 = -1.0
    history = []
    
    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss, train_acc, train_f1 = run_epoch(model, optimizer, criterion, train_loader, train_mode=True)
        val_loss, val_acc, val_f1 = run_epoch(model, optimizer, criterion, val_loader, train_mode=False)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "train_f1": train_f1,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_f1": val_f1,
        })

        print(
            f"Fold {fold+1} Epoch [{epoch}/{NUM_EPOCHS}] "
            f"train_loss={train_loss:.4f} train_f1={train_f1:.4f} | "
            f"val_loss={val_loss:.4f} val_f1={val_f1:.4f}"
        )

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), f"best_densenet50_fold_{fold+1}.pth")
            print(f"  -> Saved best model for Fold {fold+1} (val_f1={best_val_f1:.4f})")

    model = build_model()  # Re-initialize to load weights correctly
    model.load_state_dict(torch.load(f"best_densenet50_fold_{fold+1}.pth"))
    scaled_model = calibrated_model(model) 
    torch.save(scaled_model.state_dict(), f"scaled_densenet50_fold_{fold+1}.pth")
    fold_histories.append(history)
    best_fold_f1s.append(best_val_f1)
    print(f"Fold {fold+1} finished. Best F1: {best_val_f1:.4f}")

print("\n" + "="*30)
print(f"Cross Validation Complete! Mean Best F1: {np.mean(best_fold_f1s):.4f}")

Starting 5-Fold Cross Validation...

==================== FOLD 1/5 ====================


Exception ignored while calling deallocator <function _MultiProcessingDataLoaderIter.__del__ at 0x7fc094d38720>:
Traceback (most recent call last):
  File "/usr/lib/python3.14/site-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/lib/python3.14/site-packages/torch/utils/data/dataloader.py", line 1665, in _shutdown_workers
    if self._persistent_workers or self._workers_status[worker_id]:
AttributeError: '_MultiProcessingDataLoaderIter' object has no attribute '_workers_status'


Train:   0%|          | 0/620 [00:11<?, ?it/s]

Val:   0%|          | 0/155 [00:13<?, ?it/s]

Fold 1 Epoch [1/1] train_loss=2.5856 train_f1=0.1111 | val_loss=2.5136 val_f1=0.1051
  -> Saved best model for Fold 1 (val_f1=0.1051)


100%|██████████| 97/97 [00:25<00:00,  3.83it/s]


Fold 1 finished. Best F1: 0.1051

Cross Validation Complete! Mean Best F1: 0.1051


In [55]:
import pickle

with open("densenet121-training-histories.pkl", "wb") as fp:
    pickle.dump(fold_histories, fp)

In [56]:
# Validate the first trained fold model on held-out test data (X_test)
test_ds = NIHChestDataset(X_test, transform=val_tfms)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=torch.cuda.is_available(),
    num_workers=NUM_WORKERS
)

model_fold1 = build_model()
model_fold1.load_state_dict(torch.load("best_densenet50_fold_1.pth", map_location=device))
model_fold1.eval()

# Use an unweighted loss for reporting on test data
test_criterion = nn.CrossEntropyLoss()

total_loss = 0.0
steps = 0

all_preds = []
all_targets = []

# For per-class accuracy
correct_per_class = torch.zeros(nih.num_classes, device=device)
total_per_class = torch.zeros(nih.num_classes, device=device)

with torch.no_grad():
    for images, targets in test_loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        if targets.ndim > 1 and targets.size(1) > 1:
            targets_ce = torch.argmax(targets, dim=1)
        else:
            targets_ce = targets

        logits = model_fold1(images)
        loss = test_criterion(logits, targets_ce.long())

        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        total_loss += loss.item()
        steps += 1

        all_preds.append(preds.cpu().numpy())
        all_targets.append(targets_ce.cpu().numpy())

        # Per-class accuracy
        for c in range(nih.num_classes):
            mask = (targets_ce == c)
            correct_per_class[c] += (preds[mask] == targets_ce[mask]).sum()
            total_per_class[c] += mask.sum()

all_preds = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)

test_loss = total_loss / max(steps, 1)
test_acc = (all_preds == all_targets).mean()
test_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)

eps = 1e-8
per_class_acc = (correct_per_class / (total_per_class + eps)).detach().cpu().numpy()

print("=== Fold 1 Test Performance ===")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Acc : {test_acc:.4f}")
print(f"Test F1  : {test_f1:.4f}")

print("\n=== Per-class Accuracy ===")
class_names = getattr(nih, "pathologies", [f"class_{i}" for i in range(nih.num_classes)])
for i, acc in enumerate(per_class_acc):
    name = class_names[i] if i < len(class_names) else f"class_{i}"
    print(f"{name:20s}: {acc:.4f}")

=== Fold 1 Test Performance ===
Test Loss: 2.5180
Test Acc : 0.1547
Test F1  : 0.1322

=== Per-class Accuracy ===
class_0             : 0.1233
class_1             : 0.7838
class_2             : 0.0963
class_3             : 0.4444
class_4             : 0.0687
class_5             : 0.0000
class_6             : 0.3553
class_7             : 0.8462
class_8             : 0.0088
class_9             : 0.3554
class_10            : 0.4248
class_11            : 0.0323
class_12            : 0.0370
class_13            : 0.0811


In [57]:
from contextlib import contextmanager

@contextmanager
def monte_carlo_dropouts(model):
    try:
        model.eval()

        for module in model.modules():
            if module.__class__.__name__ == '_DenseLayer':
                module.training = True
        yield model
    finally:
        for module in model.modules():
            if module.__class__.__name__ == '_DenseLayer':
                module.training = False


In [ ]:
from tqdm import tqdm

def entropy(p):
    # Add epsilon to prevent log(0). 
    # Computes per-sample entropy across classes. Shape: (batch_size,)
    eps = 1e-8
    # Standard categorical entropy for softmax outputs
    categorical_ent = -p * torch.log(p + eps)
    return torch.sum(categorical_ent, dim=-1)

def epistemic_uncertainty(preds):
    # preds shape: (N_models_or_passes, batch_size, num_classes)
    
    # 1. Total entropy: entropy of the mean prediction
    p_mean = torch.mean(preds, dim=0) # (batch_size, num_classes)
    tot_ent = entropy(p_mean)         # (batch_size,)
    
    # 2. Aleatoric uncertainty: mean of the entropies
    aleatoric = torch.mean(entropy(preds), dim=0) # (batch_size,)
    
    # 3. Epistemic uncertainty
    return tot_ent - aleatoric

def compute_id_scores(id_loader, num_passes=15):
    # Pre-load models ONCE outside the batch loop to prevent massive disk I/O bottlenecks!
    models = []
    for fold in range(KFOLDS):
        m = TemperatureScaler(build_model(), device=device)
        m.load_state_dict(torch.load(f"scaled_densenet50_fold_{fold+1}.pth", map_location=device))
        m.trained = True
        models.append(m)

    id_scores = []
    with torch.no_grad():
        for images, _ in tqdm(id_loader, "Computing ID scores"):
            images = images.to(device)
            
            outputs = []
            for m in models:
                with monte_carlo_dropouts(m):
                # MCDropout: Pass data multiple times through the SAME model
                    for _ in range(num_passes):
                        logits = m(images)
                        probs = torch.softmax(logits, dim=1) # Convert logits to Multi-Class probabilities!
                        outputs.append(probs)
            
            # Stack all multi-model outputs 
            # New shape: (KFOLDS * num_passes, batch_size, num_classes)
            preds = torch.stack(outputs)
            
            # Vectorized epistemic uncertainty -> shape: (batch_size,)
            uncertainty = epistemic_uncertainty(preds)
            
            # Negative uncertainty is used so higher score = more ID (less uncertain)
            id_scores.extend(-uncertainty.cpu().numpy())
            
    return id_scores

In [59]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

test_loader = DataLoader(
    NIHChestDataset(X_test, transform=val_tfms),
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=torch.cuda.is_available(),
    num_workers=NUM_WORKERS
)

id_scores = compute_id_scores(test_loader)

Computing ID scores:   3%|▎         | 3/97 [01:24<43:55, 28.04s/it]


KeyboardInterrupt: 

In [ ]:
import pickle

# with open("id_scores.pkl", "rb") as fp:
#     id_scores = pickle.load(fp)

In [ ]:
with open("id_scores.pkl", "wb") as f:
    pickle.dump(id_scores, f)

[np.float32(-0.0070667267),
 np.float32(-0.0009613037),
 np.float32(-0.0022788048),
 np.float32(-0.0010437965),
 np.float32(-0.0018205643),
 np.float32(-0.0048947334),
 np.float32(-0.00086164474),
 np.float32(-0.002357006),
 np.float32(-0.0015234947),
 np.float32(-0.0019636154),
 np.float32(-0.004470825),
 np.float32(-0.00126791),
 np.float32(-0.006533146),
 np.float32(-0.007121086),
 np.float32(-0.0010499954),
 np.float32(-0.00091695786),
 np.float32(-0.0044231415),
 np.float32(-0.0017542839),
 np.float32(-0.00086927414),
 np.float32(-0.00093364716),
 np.float32(-0.0056295395),
 np.float32(-0.002497673),
 np.float32(-0.006664276),
 np.float32(-0.0018696785),
 np.float32(-0.0022511482),
 np.float32(-0.008825302),
 np.float32(-0.0011973381),
 np.float32(-0.00416708),
 np.float32(-0.003900528),
 np.float32(-0.0023884773),
 np.float32(-0.0017290115),
 np.float32(-0.0023732185),
 np.float32(-0.007565975),
 np.float32(-0.00093507767),
 np.float32(-0.0061836243),
 np.float32(-0.0019760132),


In [ ]:
df_ood_subset = nih.ood_df.sample(n = len(X_test), random_state=42)
ood_dataset = NIHChestDataset(df_ood_subset, transform=val_tfms)
ood_loader = DataLoader(
    ood_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=torch.cuda.is_available(),
    num_workers=NUM_WORKERS
)

In [ ]:
ood_scores = compute_id_scores(ood_loader)

Computing ID scores:   1%|          | 1/97 [00:47<1:16:13, 47.64s/it]


KeyboardInterrupt: 

In [ ]:
# with open("ood_scores.pkl", "rb") as fp:
#     ood_scores = pickle.load(fp)

In [ ]:
with open("ood_scores.pkl", "wb") as f:
    pickle.dump(ood_scores, f)

In [ ]:
from metrics import fpr
from sklearn.metrics import average_precision_score

y_true = [1] * len(id_scores) + [0] * len(ood_scores)
y_scores = id_scores + ood_scores

auprc = average_precision_score(y_true, y_scores)
fpr90 = fpr(y_true, y_scores, percentile=10)
fpr95 = fpr(y_true, y_scores, percentile=5)
fpr99 = fpr(y_true, y_scores, percentile=1)

print(f"AUPRC: {auprc:.4f}")
print(f"FPR at 90% TPR: {fpr90:.4f}")
print(f"FPR at 95% TPR: {fpr95:.4f}")
print(f"FPR at 99% TPR: {fpr99:.4f}")

NameError: name 'id_scores' is not defined

In [ ]:
from torchvision import datasets, transforms
import torch
from torch.utils.data import DataLoader, Subset

caltech_data = datasets.Caltech256(
    root='./data', 
    download=True, 
    transform=val_tfms
)

# Randomly sample the Caltech dataset to match the size of your Validation (ID) dataset
num_samples = min(len(X_test), len(caltech_data))
torch.manual_seed(42)
indices = torch.randperm(len(caltech_data))[:num_samples]
caltech_subset = Subset(caltech_data, indices.tolist())

caltech_loader = DataLoader(
    caltech_subset,
    batch_size=16,
    shuffle=False,
    pin_memory=torch.cuda.is_available(),
    num_workers=NUM_WORKERS
)

print(f"Loaded {len(caltech_subset)} Caltech256 images to use as OOD.")

Loaded 3097 Caltech256 images to use as OOD.


In [ ]:
ood_scores_caltech = compute_id_scores(caltech_loader)


Computing ID scores:   5%|▌         | 10/194 [01:42<31:22, 10.23s/it]


KeyboardInterrupt: 

In [ ]:

with open("ood_scores_caltech.pkl", "wb") as f:
    pickle.dump(ood_scores_caltech, f)

In [ ]:

# Assign class labels exactly as in the previous cell
y_true_caltech = [1] * len(id_scores) + [0] * len(ood_scores_caltech)
y_scores_caltech = id_scores + ood_scores_caltech

auprc_caltech = average_precision_score(y_true_caltech, y_scores_caltech)
fpr90_caltech = fpr(y_true_caltech, y_scores_caltech, percentile=10)
fpr95_caltech = fpr(y_true_caltech, y_scores_caltech, percentile=5)
fpr99_caltech = fpr(y_true_caltech, y_scores_caltech, percentile=1)

print("=== ID (NIH X-Ray) vs OOD (Caltech256) ===")
print(f"AUPRC: {auprc_caltech:.4f}")
print(f"FPR at 90% TPR: {fpr90_caltech:.4f}")
print(f"FPR at 95% TPR: {fpr95_caltech:.4f}")
print(f"FPR at 99% TPR: {fpr99_caltech:.4f}")

=== ID (NIH X-Ray) vs OOD (Caltech256) ===
AUPRC: 0.5968
FPR at 90% TPR: 0.8621
FPR at 95% TPR: 0.8833
FPR at 99% TPR: 0.9129
